<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left">  Agent Failure Examples  </h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

#Agent Failure Examples

**Purpose:** Show common AI agent problems **in a practical setting** and how to diagnose them using traces.  

**Concepts Covered:**
- Wrong tool selection
- Agent looping
- Premature final answer
- Using traces to debug agent behavior

#Setup
We use the classic ReAct agent implementation because it produces clear Thought → Action → Observation traces, which are ideal for teaching agent failures.

In [1]:
# Install classic LangChain + scraping libraries
!pip install -q "langchain==0.3.*" "langchain-openai==0.2.*" "langchainhub" "langchain-community==0.3.*" beautifulsoup4 requests --force-reinstall

import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("Setup complete")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.0/1

# Define Tools

Three tools are defined :

- calculate_math: Performs basic mathematical calculations using a string expression.
- get_weather: Returns dummy weather data (used to create wrong tool selection failures).
- scrape_webpage: Scrapes real content from websites. It has special logic to extract book titles and prices from books.toscrape.com.

These tools allow us to demonstrate both successful behavior and different types of agent failures in a realistic setting.

In [2]:
from langchain_core.tools import tool
import requests
from bs4 import BeautifulSoup

@tool
def calculate_math(expression: str) -> str:
    """Perform basic math calculations.
    Input should be a math expression like '15 + 7', '15 * 8'."""
    try:
        allowed = {"__builtins__": {}}
        result = eval(expression, allowed, {})
        return str(result)
    except Exception as e:
        return f"Calculation error: {str(e)}"

@tool
def get_weather(city: str) -> str:
    """Returns dummy weather for a city (for failure demos)."""
    return f" The weather in {city} is sunny and 25°C. (dummy data)"

@tool
def scrape_webpage(url: str) -> str:
    """Scrapes a webpage and extracts useful information.
    If the URL is books.toscrape.com, it extracts book titles and prices.
    Otherwise, it returns title and main content snippet."""
    try:
        headers = {"User-Agent": "Mozilla/5.0 (compatible; AI-Agent-Demo/1.0)"}
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        title = soup.title.string.strip() if soup.title else "No title"

        # Special handling for books.toscrape.com
        if "books.toscrape.com" in url:
            books = []
            for article in soup.find_all("article", class_="product_pod")[:8]:  # limit to 8 books
                book_title = article.h3.a.get("title", "No title") if article.h3 and article.h3.a else "No title"
                price_tag = article.find("p", class_="price_color")
                price = price_tag.get_text(strip=True) if price_tag else "No price"
                books.append(f"{book_title} - {price}")

            if books:
                return f"Page Title: {title}\n\nBooks found:\n" + "\n".join(books)
            else:
                return f"Page Title: {title}\nNo books found in the expected format."

        # General fallback
        paragraphs = soup.find_all("p")
        content = paragraphs[0].get_text(strip=True)[:500] if paragraphs else "No content"
        return f"Page Title: {title}\nContent snippet: {content}"

    except Exception as e:
        return f"Scraping failed: {str(e)}"

tools = [calculate_math, get_weather, scrape_webpage]

print(" Tools ready (including REAL web scraping):", [t.name for t in tools])

 Tools ready (including REAL web scraping): ['calculate_math', 'get_weather', 'scrape_webpage']


## Successful ReAct Agent (Baseline)

This cell creates a properly working ReAct agent using the standard prompt from LangChain Hub.

The agent should correctly decide which tool to use and return useful results when asked to scrape book information.

A clean prompt with explicit Action Input instructions is used to prevent common formatting errors.

In [6]:
from langchain.agents import create_react_agent, AgentExecutor
from langchain_openai import ChatOpenAI
from langchain import hub
from langchain.prompts import PromptTemplate

# Load base prompt and add strong instruction for clean Action Input
# The base_prompt variable was not used, so the problematic hub.pull call is removed.

clean_prompt_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format EXACTLY. Do not add extra text like 'url =' or explanations inside Action Input.

Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action **as a clean string only**.
             For scrape_webpage, just put the full URL like https://books.toscrape.com
             Do NOT write "url =", do NOT add quotes around the whole thing, do NOT add extra text.
Observation: the result of the action
... (this Thought/Action/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
{agent_scratchpad}"""

prompt = PromptTemplate.from_template(clean_prompt_template)

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=10,
    handle_parsing_errors=True
)

print("Successful ReAct agent created")

# Clean test query
response = agent_executor.invoke({"input": "List a few books with their prices from https://books.toscrape.com"})
print("\nFinal Answer:", response["output"])

Successful ReAct agent created


> Entering new AgentExecutor chain...
Thought: I need to scrape the webpage to extract book titles and prices.
Action: scrape_webpage
Action Input: https://books.toscrape.comPage Title: All products | Books to Scrape - Sandbox

Books found:
A Light in the Attic - Â£51.77
Tipping the Velvet - Â£53.74
Soumission - Â£50.10
Sharp Objects - Â£47.82
Sapiens: A Brief History of Humankind - Â£54.23
The Requiem Red - Â£22.65
The Dirty Little Secrets of Getting Your Dream Job - Â£33.34
The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull - Â£17.93I have successfully extracted book titles and prices.
Final Answer: A Light in the Attic - £51.77, Tipping the Velvet - £53.74, Soumission - £50.10, Sharp Objects - £47.82, Sapiens: A Brief History of Humankind - £54.23, The Requiem Red - £22.65, The Dirty Little Secrets of Getting Your Dream Job - £33.34, The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Wo

## 1. Wrong Tool Selection

In this example, a bad system instruction is added to the prompt.

The instruction forces the agent to always start with the get_weather tool whenever a URL or website question appears.

You can observe in the trace how the agent incorrectly chooses the weather tool instead of the scrape_webpage tool, even though scraping is clearly needed.

In [7]:
from langchain.prompts import PromptTemplate

# Fixed prompt - includes ALL required variables: {tools}, {tool_names}, {agent_scratchpad}
bad_prompt_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

CRITICAL RULE: For ANY question that contains a URL or asks for website content, you MUST start by calling the get_weather tool first.

Use the following format exactly:

Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
{agent_scratchpad}"""

bad_prompt = PromptTemplate.from_template(bad_prompt_template)

bad_agent = create_react_agent(llm=llm, tools=tools, prompt=bad_prompt)
bad_executor = AgentExecutor(
    agent=bad_agent,
    tools=tools,
    verbose=True,
    max_iterations=6,
    handle_parsing_errors=True
)

print(" Wrong Tool Selection Demo")
response = bad_executor.invoke({"input": "What is the title of the first book on https://books.toscrape.com"})
print("\nFinal Output:", response.get("output", "No output"))

 Wrong Tool Selection Demo


> Entering new AgentExecutor chain...
Thought: I should start by calling the get_weather tool
Action: get_weather
Action Input: any city The weather in any city is sunny and 25°C. (dummy data)Now I can use the scrape_webpage tool to get the title of the first book on the website
Action: scrape_webpage
Action Input: https://books.toscrape.comPage Title: All products | Books to Scrape - Sandbox

Books found:
A Light in the Attic - Â£51.77
Tipping the Velvet - Â£53.74
Soumission - Â£50.10
Sharp Objects - Â£47.82
Sapiens: A Brief History of Humankind - Â£54.23
The Requiem Red - Â£22.65
The Dirty Little Secrets of Getting Your Dream Job - Â£33.34
The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull - Â£17.93The title of the first book on https://books.toscrape.com is "A Light in the Attic"
Final Answer: A Light in the Attic

> Finished chain.

Final Output: A Light in the Attic


## 2. Agent Looping

This section demonstrates agent looping behavior.

A strict rule is added that forces the agent to call a tool on every single step and forbids it from giving a Final Answer until at least 12 tool calls have been made.

In the trace, you will see the same Thought → Action → Observation pattern repeating many times until the maximum iteration limit is reached.

In [8]:
from langchain.prompts import PromptTemplate

# Strong looping prompt
loop_prompt_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

CRITICAL RULE - STRICT LOOP MODE:
You MUST call a tool on **every single step**.
You are **strictly forbidden** from giving any Final Answer until you have made **at least 12 tool calls**.
Keep calling tools even if the results are not helpful. Do not stop early.

Use the following format exactly:

Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Observation must repeat many times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
{agent_scratchpad}"""

loop_prompt = PromptTemplate.from_template(loop_prompt_template)

loop_agent = create_react_agent(llm=llm, tools=tools, prompt=loop_prompt)

loop_executor = AgentExecutor(
    agent=loop_agent,
    tools=tools,
    verbose=True,
    max_iterations=18,
    handle_parsing_errors=True,
    early_stopping_method="force"
)

print(" Agent Looping Demo")

response = loop_executor.invoke({"input": "Find the price of the most expensive book on https://books.toscrape.com"})

print("\nFinal Output:", response.get("output", "Agent stopped due to max iterations"))

 Agent Looping Demo


> Entering new AgentExecutor chain...
Thought: I need to scrape the webpage to extract book titles and prices
Action: scrape_webpage
Action Input: https://books.toscrape.comPage Title: All products | Books to Scrape - Sandbox

Books found:
A Light in the Attic - Â£51.77
Tipping the Velvet - Â£53.74
Soumission - Â£50.10
Sharp Objects - Â£47.82
Sapiens: A Brief History of Humankind - Â£54.23
The Requiem Red - Â£22.65
The Dirty Little Secrets of Getting Your Dream Job - Â£33.34
The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull - Â£17.93I have extracted the book titles and prices, now I need to find the most expensive book.
Action: calculate_math
Action Input: max(51.77, 53.74, 50.10, 47.82, 54.23, 22.65, 33.34, 17.93)Calculation error: name 'max' is not definedI need to use the correct syntax for finding the maximum value.
Action: calculate_math
Action Input: max(51.77, 53.74, 50.10, 47.82, 54.23, 22.65, 33.34, 17.93)Calculation 

## 3. Premature Final Answer

This example shows premature final answer behavior.

The prompt contains a strong instruction telling the agent that it has excellent built-in knowledge and should answer website or book questions immediately without calling any tools.

In the trace, you will see the agent jumping directly to Final Answer with very few or no tool calls.

In [9]:
from langchain.prompts import PromptTemplate

premature_prompt_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

CRITICAL RULE: You have very strong built-in knowledge about books and websites.
If the question is about books or any website like books.toscrape.com,
you should give the Final Answer IMMEDIATELY without calling any tools.

Do NOT call any tools. Answer directly from your knowledge.

Use the following format exactly:

Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
{agent_scratchpad}"""

premature_prompt = PromptTemplate.from_template(premature_prompt_template)

premature_agent = create_react_agent(llm=llm, tools=tools, prompt=premature_prompt)

premature_executor = AgentExecutor(
    agent=premature_agent,
    tools=tools,
    verbose=True,
    max_iterations=5,
    handle_parsing_errors=True
)

print("Premature Final Answer Demo")
response = premature_executor.invoke({"input": "What books are listed on https://books.toscrape.com?"})
print("\nFinal Output:", response.get("output", "No output"))

Premature Final Answer Demo


> Entering new AgentExecutor chain...
Thought: I have strong built-in knowledge about books and websites like books.toscrape.com.
Final Answer: I now know the final answer
Final Answer: The books listed on https://books.toscrape.com are extracted using the scrape_webpage tool.

> Finished chain.

Final Output: The books listed on https://books.toscrape.com are extracted using the scrape_webpage tool.


## 4. Bad Tool Input During Scraping

This section demonstrates a common real-world failure where the agent calls the correct tool but passes invalid input.

The prompt forces the agent to always pass the string "invalid-url" to the scrape_webpage tool.

you can clearly see in the trace how the agent repeatedly sends bad input, the tool returns an error in the Observation, and the agent eventually fails to complete the task.

In [10]:
from langchain.prompts import PromptTemplate

bad_input_prompt_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

CRITICAL RULE (MUST FOLLOW):
When using the scrape_webpage tool, you MUST ALWAYS pass the string "invalid-url" as the Action Input.
Never use the real URL provided in the question.
Always use exactly "invalid-url" — no matter what.

Use the following format exactly:

Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
{agent_scratchpad}"""

bad_input_prompt = PromptTemplate.from_template(bad_input_prompt_template)

bad_input_agent = create_react_agent(llm=llm, tools=tools, prompt=bad_input_prompt)

bad_input_executor = AgentExecutor(
    agent=bad_input_agent,
    tools=tools,
    verbose=True,
    max_iterations=8,
    handle_parsing_errors=True
)

print(" Bad Tool Input (Scraping Failure) Demo")
response = bad_input_executor.invoke({"input": "What books are listed on https://books.toscrape.com?"})
print("\nFinal Output:", response.get("output", "No output"))

 Bad Tool Input (Scraping Failure) Demo


> Entering new AgentExecutor chain...
Thought: We need to scrape the webpage to extract the book titles and prices.
Action: scrape_webpage
Action Input: "invalid-url"Scraping failed: Invalid URL 'invalid-url': No scheme supplied. Perhaps you meant https://invalid-url?We need to use the correct format for the URL when scraping.
Action: scrape_webpage
Action Input: "invalid-url"Scraping failed: Invalid URL 'invalid-url': No scheme supplied. Perhaps you meant https://invalid-url?We need to specify the correct URL for scraping.
Action: scrape_webpage
Action Input: "invalid-url"Scraping failed: Invalid URL 'invalid-url': No scheme supplied. Perhaps you meant https://invalid-url?We need to remember to always use "invalid-url" as the Action Input when scraping.
Final Answer: Scraping failed: Invalid URL 'invalid-url': No scheme supplied. Perhaps you meant https://invalid-url?

> Finished chain.

Final Output: Scraping failed: Invalid URL 'invalid-url'

## 5. Using Traces to Debug Agent Behavior

The verbose=True output shows the complete ReAct reasoning loop for every agent run.

Key elements in the trace are:

- Thought: The agent's reasoning and plan
- Action: The tool the agent decided to use
- Action Input: The exact input sent to the tool
- Observation: The result or error returned by the tool
- Final Answer: The final response given to the user

By carefully reading these traces, You can identify:
- Wrong tool selection (incorrect Action)
- Agent looping (repeated patterns)
- Premature final answer (missing tool calls)
- Bad tool input (errors visible in Observation)

## Conclusion

This notebook has demonstrated four common agent failures:

1. Wrong tool selection
2. Agent looping
3. Premature final answer
4. Bad tool input during scraping

Each failure was intentionally created using modified prompts, and the verbose traces were used to diagnose the problems.

You should now be able to recognize these issues in real agent systems and understand how to fix them by improving prompts and tool descriptions.

Practice by modifying the bad prompts in each section and comparing the resulting traces.